# Core NLP — Natural Language Processing Basics (with STAR)

*A beginner-to-interview notebook covering **tokenisation in detail**, text cleaning /
normalisation, Bag-of-Words, **TF-IDF**, n-grams, and text similarity — every concept wrapped
in the **Situation → Task → Action → Result (STAR)** template, plus **Problem → Solution →
Alternatives → Drawbacks** so you can *defend* your choices in an interview.*

> Runs **entirely offline** on fake data with libraries already in the `qf` env
> (scikit-learn, scipy, pandas, numpy, matplotlib) — nothing to download.


## Before we start — the STAR + P-S-A-D "recipe" you will reuse

Every section uses two paired structures. Learn the pattern once, reuse it everywhere.

**⭐ STAR = how you *describe* an ML task:**
| Field | What you say |
|---|---|
| **S**ituation | A real business/data problem. |
| **T**ask | The ML formulation (what we predict / structure we find). |
| **A**ction | The exact steps: preprocessing → model → validation. |
| **R**esult | **Numeric success criteria** — how do we *know* it worked? |

**🧭 P-S-A-D = how you *defend* a choice:**
- **Problem** — why the naive approach isn't enough.
- **Solution** — the technique you pick.
- **Alternatives** — what else you could have used, and *when* each is better.
- **Drawbacks** — the honest weaknesses interviewers probe.

*Beginner tip: writing the success criteria (Result) FIRST forces you to think about what
"working" means before you code — exactly what an interviewer wants to hear.*


In [ ]:
# --- Setup: imports + display helpers ---
%matplotlib inline
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)

print("Setup ready. Everything in this notebook uses FAKE text + built-in libraries.")


## 1. Natural Language Processing (NLP) — the big picture

### 🧠 Beginner's guide
Language is **unstructured**: raw text is a stream of characters, not a neat table of numbers.
Machine learning needs **numbers**, so the first job of every NLP pipeline is:

> raw text ➜ clean & tokenise ➜ turn words into numbers (features) ➜ learn/predict

NLP covers many tasks, all of which you can group by whether they use labels:
- **Supervised** NLP — classify (spam/review sentiment), tag parts-of-speech, named-entity
  recognition (NER), machine translation.
- **Unsupervised** NLP — topic modelling, clustering documents, word embeddings (word2vec/BERT).
- **Similarity/Retrieval** — "find documents most like this one" (search, chatbots).

The **atomic unit** of all of these is the *token* — and that is why we spend a full section on
tokenisation before touching any algorithm.

### ⭐ STAR — an NLP pipeline (the reusable "whole game" example)
| Field | Content |
|---|---|
| **S**ituation | A company gets thousands of support emails / product reviews every day and must triage them automatically. |
| **T**ask | Build a pipeline that turns English text into predictive features and classifies each item. |
| **A**ction | Clean text → **tokenise** → normalise (lowercase, remove stopwords, stem/lemmatise) → vectorise (**TF-IDF**) → fit a classifier → score on a held-out set. |
| **R**esult (**success criteria**) | Accuracy/F1 above a business threshold on un-seen text; the model depends on meaningful words, not junk; the pipeline is fast enough and the choice of each step is *defensible*. |


In [ ]:
# --- 1. The journey from raw text to numbers ----
raw = "Hello!! The QUICK brown fox jumps over the lazy dog, twice!!"
print("RAW text :", repr(raw))
print("Goal: turn this messy string into clean numeric features a model can learn from.")


## 2. Tokenisation in detail 🧠 (interview centrepiece)

**Tokenisation** = chopping text into *tokens* (usually words, but they can be sub-words,
characters, sentences). It is the first and most error-prone step — everything after it
inherits its mistakes.

### 🧭 P-S-A-D — Tokenisation
- **Problem** — punctuation, casing, names, chains like `"don't"`, hashtags, URLs,
  `$9.99`, `2.5GB` all break the naive "split on spaces" approach. A spam detector that reads
  `"win!!!"` and `"win"` as different words loses crucial signal.
- **Solution** — start with a **regular-expression word tokeniser** (`\b[\w']+\b`) that keeps
  letters/apostrophes and drops punctuation, applied after **lowercasing**. It's fast, simple,
  and works everywhere (no extra packages needed).
- **Alternatives** —
  - *Whitespace `split()`* — only OK for already-clean text.
  - *Sentence tokenisation* (nltk `sent_tokenize`, spaCy) — needed when each *sentence* is the unit.
  - *Sub-word / BPE* (Byte-Pair Encoding, used by BERT/GPT) — splits rare words into common
    pieces (`tokenization ➜ token + ization`), so the model never meets an unknown word.
- **Drawbacks** — regex tokenisers don't understand *language*: they break on contractions,
  emojis, "3rd", or multiple languages. For production NLP, use a real tokenizer (spaCy /
  HuggingFace) instead of hand-rolled regex.

### ⭐ STAR — Tokenisation
| Field | Content |
|---|---|
| **S**ituation | We must feed words to a model, but raw text has mixed casing, punctuation, and noise that would explode the vocabulary. |
| **T**ask | Split text into clean, consistent tokens with minimal loss of meaning. |
| **A**ction | Lowercase, then extract word tokens with a regex, optionally remove stopwords, count unique tokens (the vocabulary). |
| **R**esult (**success criteria**) | A bounded vocabulary of meaningful tokens; "run"/"RUN"/"run," map to the same token; the token stream per document is inspectable and repeatable. |


In [ ]:
# --- 2. Three tokenisation flavours on the same sentence ----
raw = "The QUICK brown fox jumps over the lazy dog! Shouldn't we tokenise it?"
print("RAW      :", repr(raw))
print()
# (a) whitespace splitting - naive, keeps punctuation glued to words
print("1) split()          :", raw.split())
# (b) regex word tokenisation + lowercase - keeps words/apostrophes, drops punctuation
tokens = re.findall(r"\b[\w']+\b", raw.lower())
print("2) regex + lower    :", tokens)
# (c) same regex, original case preserved
print("3) regex (no lower) :", re.findall(r"\b[\w']+\b", raw))
print()
print("Notice: 'Shouldn't' stays ONE token in the regex version - better than splitting.")


In [ ]:
# --- 2b. From tokens to a vocabulary + stopword removal ----
corpus = [
    "The quick brown fox jumps over the lazy dog",
    "A quick glimpse of a fox by the pond",
    "Brown bears are quick and powerful animals",
]
print("Corpus documents:", len(corpus))
# hand-check the first document's tokens
toks = re.findall(r"\b[\w']+\b", corpus[0].lower())
print("doc1  tokens:", toks)

# build the full vocabulary across the corpus
STOP = set("the a of and by is are".split())
full_tokens = [re.findall(r"\b[\w']+\b", d.lower()) for d in corpus]
vocab = sorted({t for d in full_tokens for t in d})
print("Vocabulary size (all unique words):", len(vocab), "->", vocab)

# stopwords ('the','a','of','and','by') are common but carry little class signal
for i, doc_toks in enumerate(full_tokens, 1):
    kept = [t for t in doc_toks if t not in STOP]
    print(f"doc{i} content words:", kept)
print("Success criteria met: shared tokens like 'quick'/'fox' are easy to compare;")
print("stopwords removed so common filler does not dominate the count.")


## 3. Normalisation: stemming & lemmatisation

Tokens like `running`, `runner`, `runs` are *different spellings of the same idea*. This
increases vocabulary size and creates redundant features.

### 🧭 P-S-A-D — Normalisation
- **Problem** — `jump`, `jumps`, `jumped`, `jumping` are 4 separate columns in BoW, so the
  model must "spend" capacity learning they mean the same thing.
- **Solution** — **stemming** (crude suffix chopping). It is cheap and needs no dictionary.
- **Alternatives** — **lemmatisation** (e.g. WordNet via nltk/spaCy) returns a real dictionary
  word (`better ➜ good`, `ran ➜ run`) using Part-of-Speech context, and is more accurate at
  the cost of an external dictionary model.
- **Drawbacks** — stemming is lossy (`running ➜ runn`, `studies ➜ studi`) and can merge
  unrelated words (`university/universal`). Lemmatisation is slower and needs a model.


In [ ]:
# --- 3. A tiny rule-based stemmer (educational stand-in for Porter Stemmer) ----
def simple_stem(word):
    word = word.lower()
    for suf in ("ing", "ed", "ies", "es", "ly", "s"):   # try suffixes from longest to shortest
        if word.endswith(suf) and len(word) > len(suf) + 2:
            return word[: -len(suf)]
    return word

words = ["running", "jumps", "jumped", "studies", "happily", "cats", "quick", "fox"]
for w in words:
    print(f"{w:10s} -> {simple_stem(w):10s}   (real Porter would do even better: "
          f"running->run, studies->studi)")


## 4. Bag of Words (BoW) — text as a numeric matrix 🧠

Once we have clean tokens, the classic first "vectorisation" method is **Bag of Words**:
for each document, count how often each vocabulary word appears.

> It's called a *bag* because **word order is thrown away** — "the dog chased the cat" and
> "the cat chased the dog" produce the *same* bag.

### 🧭 P-S-A-D — Bag of Words
- **Problem** — a model needs one fixed-size numeric vector per document, but text varies in
  length and vocabulary.
- **Solution** — build a matrix `(documents × vocabulary)` of word **counts** (`CountVectorizer`).
- **Alternatives** — binary indicators (is the word present?) when only presence matters; TF-IDF
  (next section) to down-weight common words; embeddings (word2vec/BERT) for word *meaning*.
- **Drawbacks** — BoW ignores order and negation (`"not good"` = `"good"`), can be huge and
  sparse, and gives equal weight to trivial words.


In [ ]:
# --- 4. Bag of Words: count vocabulary terms per document ----
docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs make great pets",
]
vec = CountVectorizer()                     # default: lowercase, tokenise, keep all words
X = vec.fit_transform(docs)
print("Vocabulary:", vec.get_feature_names_out())
df = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out(), index=["doc1", "doc2", "doc3"])
display(df)
print()
print("Read: doc1 contains 'the' twice, 'cat' once, 'mat' once, and no 'dog'.")


## 5. TF-IDF — the star of text features ⭐ (interview favourite)

**Bag of Words** counts words, but *common* words ("the", "and") then dominate. **TF-IDF**
fixes this by asking two questions per word:

- **TF (term frequency)** — how important is the word *in this document*? → *high* if it repeats here.
- **IDF (inverse document frequency)** — how *rare* is it across all documents? → words that
  appear in *few* documents get a *high* IDF.

**IDF formula** (commonly `log((N + 1) / (df + 1)) + 1`, sklearn adds 1 for smoothing):

```
IDF(word) = log( total_documents / number_of_documents_containing_word )
```

So a word that is common in *this* doc but rare in the *collection* scores highest —
exactly the words that best tell documents apart.

> ⚠️ **Interview gotcha:** sklearn's `TfidfVectorizer` by default **L2-normalises each row**
> (divides by document length) so the displayed numbers are not *raw* tf×idf — they are scaled
> so every document vector has unit length. That's convenient for cosine similarity.

### 🧭 P-S-A-D — TF-IDF
- **Problem** — BoW's raw counts over-weight filler words and are biased by document length.
- **Solution** — weight each feature by `tf × idf`.
- **Alternatives** — hashing trick (hash vectorizer) to save memory; sub-linear TF (log-transform)
  to further tame long documents; embeddings when you want *meaning* not exact-match counts.
- **Drawbacks** — still a **bag** (order/negation ignored), and IDF is computed *only on the
  training corpus* — you fit it on train and only transform test, never fit/test together
  (a classic **data-leakage** interview trap).


In [ ]:
# --- 5a. TF-IDF: weights that reward rare-but-informative words ----
documents = [
    "the quick brown fox jumps over the lazy dog",
    "quick brown fox runners love long distance running",
    "the stock market fell sharply and investors panicked",
    "market analysts study the stock market every single day",
]
tfidf = TfidfVectorizer(stop_words="english")          # drop 'the' etc. automatically
Xti = tfidf.fit_transform(documents)                   # fit on the corpus (training set!)

df = pd.DataFrame(Xti.toarray(),
                  columns=tfidf.get_feature_names_out(),
                  index=["d1 fox", "d2 running", "d3 stock", "d4 market"]).round(3)
display(df.T)                                          # transpose: rows = terms, cols = docs
print()
print("IDF per term  (rarer across docs = higher weight):")
for w, v in zip(tfidf.get_feature_names_out(), tfidf.idf_):
    print(f"   {w:10s} {v:6.3f}")
print("Notice 'fox'/'running'/'stock' (rare) weigh far more than the untracked stopwords.")


In [ ]:
# --- 5b. Visualise: top words per document by TF-IDF weight ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, doc_idx, title in zip(axes, [0, 2], ["doc d1 (fox article)", "doc d3 (stock crash)"]):
    weights = pd.Series(Xti.toarray()[doc_idx], index=tfidf.get_feature_names_out())
    top = weights.sort_values(ascending=False).head(6)
    ax.barh(top.index[::-1], top.values[::-1], color="#2b6cb0")
    ax.set_title(title)
    ax.set_xlabel("TF-IDF weight")
plt.tight_layout()
plt.show()
print("Result: each document is summarised by its few highest-weight words - no filler.")


## 6. Text similarity: cosine similarity over TF-IDF 🧠

"Which FAQ answer matches this question?" — chatbots and search live on **similarity** between
two TF-IDF vectors.

The **cosine similarity** between two vectors `a` and `b` is the cosine of the angle between
them:

```
cos(a, b) = (a·b) / (|a|·|b|)      -> 1 = same direction (similar), 0 = unrelated
```

Because TF-IDF rows are length-normalised, the dot product already IS the cosine similarity.

### 🧭 P-S-A-D — Text similarity
- **Problem** — documents have different lengths; raw overlap is a poor similarity measure.
- **Solution** — represent each doc by its TF-IDF vector and measure cosine similarity.
- **Alternatives** — Jaccard similarity at the token-set level (fast, ignores frequency);
  overlap coefficient; dense embeddings (nearest-neighbour over BERT vectors) when you want
  *semantic* similarity beyond shared words.
- **Drawbacks** — cosine on TF-IDF matches **shared words only**: "car repair" vs "auto fix"
  get a score of zero because no term overlaps, even though they mean the same.


In [ ]:
# --- 6. Cosine similarity between the TF-IDF documents ----
sim = cosine_similarity(Xti)
sim_df = pd.DataFrame(sim, index=["d1 fox", "d2 running", "d3 stock", "d4 market"],
                      columns=["d1 fox", "d2 running", "d3 stock", "d4 market"]).round(3)
display(sim_df)
print()
plt.figure(figsize=(6, 5))
sns.heatmap(sim, annot=True, cmap="Blues", fmt=".2f",
            xticklabels=["d1 fox", "d2 running", "d3 stock", "d4 market"],
            yticklabels=["d1 fox", "d2 running", "d3 stock", "d4 market"])
plt.title("Document similarity heatmap (1.0 = identical direction)")
plt.show()
print("Result: the two finance docs (d3, d4) are most similar (share 'market'/'stock'),")
print("which matches their subject matter - the metric is sensible.")


## 7. N-grams: reintroducing word order (a little)

**Unigrams** (single words) ignore order. **N-grams** take sequences of N words as features,
so word-order *near* a word is captured.

### 🧭 P-S-A-D — N-grams
- **Problem** — `"not good"` and `"very good"` are identical in unigram BoW; order matters.
- **Solution** — use `bigram` (2-word) or `trigram` features via `ngram_range=(1, 2)`.
- **Alternatives** — character n-grams (robust to typos/morphology); word n-grams up to 3;
  sub-word embeddings if you need full context.
- **Drawbacks** — vocabulary explodes ($V^n$ possible combos) and most n-grams are rare; use
  n-grams sparingly and consider `min_df` to prune ultra-rare ones.


In [ ]:
# --- 7. Unigram vs Bigram features ----
big = CountVectorizer(ngram_range=(1, 2))
Xb = big.fit_transform(docs)                      # reuse the 3-doc 'cat/dog/pets' corpus
dfb = pd.DataFrame(Xb.toarray(), columns=big.get_feature_names_out(),
                   index=["doc1", "doc2", "doc3"])
display(dfb)
print("Now 'the cat' and 'sat on' carry meaning together - order is partly restored.")


## 8. End-to-end mini-project: TF-IDF ➜ classifier (supervised NLP) 🚀

Time to glue it together exactly like a real task: **detect spam from ham**.

### ⭐ STAR — Spam classifier
| Field | Content |
|---|---|
| **S**ituation | Inbound messages mix genuine requests with spam; manually screening them wastes analyst time. |
| **T**ask | Classify each message `spam (1)` vs `ham (0)` from its text. |
| **A**ction | Tokenise → TF-IDF (fit on *train only*!) → train `LogisticRegression` → evaluate on a held-out set with accuracy + confusion matrix. |
| **R**esult (**success criteria**) | Test accuracy clearly above random (50%); predicted labels agree with labels on the holdout; the most *important* words are genuinely spammy/benign. |


In [ ]:
# --- 8. Spam vs ham with TF-IDF + Logistic Regression ----
spam = [
    "buy now cheap pills limited offer", "click here to claim your free prize",
    "you have won a million dollars click", "limited time sale huge discount today only",
    "act now before this offer ends quickly", "guaranteed income work from home easy cash",
    "urgent! your account needs verification now", "for only a small fee unlock premium access",
    "double your money with this secret strategy", "last chance free gift card inside",
    "make thousands while you sleep no effort", "special offer just for you hot deal",
]
ham = [
    "meeting scheduled for three this afternoon", "please review the attached report today",
    "the quarterly earnings call is on wednesday", "can you send the updated budget by friday",
    "thank you for your help on the project", "lunch booked at noon near the office",
    "reminder to update the project tracker", "the client approved our proposal yesterday",
    "let us discuss the strategy after lunch", "i attached the revised spreadsheet",
    "team standup moved to ten o'clock", "please confirm your availability for tuesday",
]
docs = spam + ham
y = np.array([1] * len(spam) + [0] * len(ham))

vec = TfidfVectorizer(stop_words="english")   # tokenise + TF-IDF in one object
X = vec.fit_transform(docs)                   # fit vectoriser on ALL of this small demo set
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)

print("Test accuracy:", round(accuracy_score(yte, pred), 3))
ConfusionMatrixDisplay(confusion_matrix(yte, pred), display_labels=["ham (0)", "spam (1)"]).plot()
plt.title("Spam-vs-Ham confusion matrix (TF-IDF + Logistic Regression)")
plt.show()

coef = pd.Series(clf.coef_[0], index=vec.get_feature_names_out()).sort_values()
print("Most 'spammy' words (largest positive weights):", list(coef.tail(5).index))
print("Most 'benign'  words (most negative weights):  ", list(coef.head(5).index))


### Discussion — the mini-project (interview favourites)
- **"Why TF-IDF + Logistic Regression?"** → Fast, sparse-friendly, and the coefficients double
  as *feature importances* (the table above IS your explanation to business). No black box.
- **"Where is the leakage trap here?"** → When building a *real* system you must `fit` the
  vectoriser on the **training** set only. If you fit it on the *whole* dataset before the
  split, IDF "peeks" at test documents and inflates test scores — the same leakage discussion
  as any preprocessing scaler.
- **"How would you handle class imbalance?"** → Use precision/recall/F1 or ROC-AUC instead of
  accuracy, and consider class weights or resampling. Accuracy alone is misleading when 95% of
  messages are ham.
- **"When would you switch to an embedding / transformer?**" → When items share *meaning* but
  not vocabulary (synonyms, paraphrases) — e.g. "car repair" vs "auto fix". TF-IDF can't see
  that without overlapping words; a sentence embedding can.


## 9. Case study: payment confirmations in a shared inbox 💳

**The real scenario:** a client sends a request to *confirm a payment was done*. The
confirmation emails land in a **shared mailbox** alongside many others. Our job:

> Extract **client information from all the emails** and **link each one to the performed
> transaction** for the payment-service request.

This is classic **information extraction + record linking (entity resolution)** — the moment
all the earlier ideas (tokenisation, regex, TF-IDF, cosine similarity) combine into one
workflow.

### ⭐ STAR — link emails to transactions
| Field | Content |
|---|---|
| **S**ituation | Payment confirmations arrive as free text in a shared inbox: client name, ref, amount & date are mixed into prose, and the client asks us to confirm against the ledger. |
| **T**ask | Turn each email into structured fields and link it to the matching transaction on the payment service. |
| **A**ction | Tokenise/clean each email; **extract** fields (txn ref, amount, currency, date, client) with regex rules; build the transactions ledger; **join** (exact on txn ref, fuzzy on client name via TF-IDF cosine as a fallback). |
| **R**esult (**success criteria**) | Every email links to exactly one ledger row; amount/date from the email **agree with the ledger**; anything ambiguous or unmatched is flagged for manual review. |

### 🧭 P-S-A-D — extraction + linking
- **Problem** — key data is buried in unstructured text, and inboxes mix senders; a naive
  keyword search can't join to the payments system.
- **Solution** — regex field extraction into a table, then a database-style join on the
  transaction reference — with a **TF-IDF cosine fallback** when a client name differs slightly.
- **Alternatives** — a trained NER model (spaCy / transformers) for wildly varying formats;
  fuzzy-string libraries (RapidFuzz); dense embeddings for semantic matching.
- **Drawbacks** — regex rules fail on unseen formats; refs could be reused or mistyped;
  over-confident joins risk paying the wrong invoice. Always **validate amount + date** and keep
  unmatched rows for a human.


In [ ]:
# --- 9a. Parse the shared 'payment confirmation' inbox into structured fields ----
emails = [
 "Payment confirmation - Acme Trading Ltd\nHi team, please confirm our wire transfer "
 "of USD 1,250.00 (ref TXN-482913) dated 03/10/2026.",
 "Payment confirmation - Bakersfield GmbH\nDear support, kindly confirm the payment of "
 "EUR 3,400.00 with reference TXN-710284 on 03/10/2026.",
 "Payment confirmation - Canvas Retail Co.\nHello, we need confirmation for GBP 870.50 "
 "remittance ref TXN-551927 dated 04/10/2026.",
 "Payment confirmation - Delta Freight NV\nPlease confirm receipt of USD 990.00, transaction "
 "TXN-834206, dated 05/10/2026.",
 "Payment confirmation - Elm Ventures\nConfirm the transfer of EUR 12,000.00 for ref "
 "TXN-112340 on 06/10/2026 please.",
 "Payment confirmation - Ferra Ltd\nKindly confirm our USD 2,300.00 payment, tx ref "
 "TXN-991788, dated 07/10/2026.",
]

def parse_email(text):
    client = re.search(r"Payment confirmation - (.+)[\n]", text).group(1)   # the client name
    ref    = re.search(r"TXN-\d+", text).group(0)                            # transaction ref
    cur    = re.search(r"(USD|EUR|GBP)", text).group(1)                      # currency
    amt    = float(re.search(r"([\d,]+\.\d{2})", text).group(1).replace(",", ""))
    date   = re.search(r"\d{2}/\d{2}/\d{4}", text).group(0)
    return ref, client, cur, amt, date

inbox = pd.DataFrame((parse_email(e) for e in emails),
                     columns=["txn_id", "client", "currency", "amount", "date"])
inbox

In [ ]:
# --- 9b. Link each email to its transaction on the payment service ----
transactions = pd.DataFrame([
    ["TXN-482913", "Acme Trading Ltd.", "USD", 1250.00, "03/10/2026", "completed"],
    ["TXN-710284", "Bakersfield GmbH",  "EUR", 3400.00, "03/10/2026", "completed"],
    ["TXN-551927", "Canvas Retail Co.",  "GBP",  870.50, "04/10/2026", "completed"],
    ["TXN-834206", "Delta Freight NV",  "USD",  990.00, "05/10/2026", "pending"],
    ["TXN-112340", "Elm Ventures PLC",  "EUR",12000.00, "06/10/2026", "completed"],
    ["TXN-991788", "Ferra Ltd",         "USD", 2300.00, "07/10/2026", "failed"],
], columns=["txn_id", "client", "currency", "amount", "date", "status"])

linked = inbox.merge(transactions, on="txn_id", how="left",
                     suffixes=("_email", "_ledger"))
linked["amount_matches"] = linked["amount_email"] == linked["amount_ledger"]
print(f"Emails linked to a transaction: {linked['status'].notna().sum()} / {len(linked)}")
print(f"Rows where email amount matches the ledger: {int(linked['amount_matches'].sum())} / {len(linked)}")
linked

In [ ]:
# --- 9c. Fallback: link by CLIENT NAME with TF-IDF cosine (if the txn ref is missing) ----
email_names = inbox["client"].tolist()
ledger_names = transactions["client"].tolist()

vec = TfidfVectorizer(token_pattern=r"\b\w+\b")      # reuse TF-IDF to compare two short names
M = vec.fit_transform(email_names + ledger_names)
S = cosine_similarity(M[:len(email_names)], M[len(email_names):])      # rows=emails, cols=ledger

name_match = pd.DataFrame({
    "email_client": email_names,
    "best_ledger_match": [ledger_names[i] for i in S.argmax(axis=1)],
    "score": S.max(axis=1).round(3),
})
name_match

### Discussion — the payments case (how this maps to an interview answer)
- **"How do you confirm the payment was really done?"** → The email gives me the client's ref,
  amount and date; I **join to the ledger** and check amount and date agree (see `amount_matches`).
  Agreement on two independent fields is strong evidence; a mismatch flags the email for review.
- **"What if the client name differs across emails ('Acme Trading' vs 'Acme Trading Ltd.')?"**
  → That is **entity resolution**. TF-IDF + cosine similarity treats the two strings as close
  even when punctuation differs; the fallback in 9c returns the best ledger match with a score.
- **"Why not just do a keyword search for the invoice number?"** → Tokenisation strips
  punctuation and regex finds the real ref wherever it appears in prose; then we link on it.
  A keyword search breaks on written-out refs or a ref that appears in the subject line instead.
- **"What are the risks?"** → False joins (same ref, wrong amount), missed emails (new format =
  regex fails), and shared-inbox noise. So: validate on amount + date, flag low-confidence /
  unmatched rows for a human, and prefer a trained NER when the formats vary a lot.


## 10. Cheat-sheet, common traps & interview recap

### Choose-your-technique table
| Task (⭐ Situation) | Technique (⭐ Action) | Success criteria (⭐ Result) | When to switch **away** (Drawback) |
|---|---|---|---|
| Split text into words | Regex tokenise → lowercase | Clean, repeatable tokens | Production language → spaCy/HF |
| Reduce variance of same word | Stem / Lemmatise | Smaller vocab, no meaning loss | Stemming too lossy → Lemmatise |
| Words ➜ numbers | Bag of Words (counts) | Fixed matrix, model-ready | Need meaning/semantics → embeddings |
| Down-weight filler words | **TF-IDF** | Top words are informative | No word overlap → embeddings |
| Are two texts similar? | Cosine over TF-IDF | Sensible ranking | Synonyms → dense embeddings |
| Order matters | N-grams (1–2) | Better accuracy on phrase tasks | Vocabulary exploding → keep short |
| Classify text (supervised) | TF-IDF + Logistic Reg | Clear accuracy/F1, meaningful coefs | Complex/negation → SVM/transformers |

### ⭐ The reusable STAR "elevator pitch" for any NLP question
> **S**ituation (real text problem) → **T**ask (classify / retrieve / cluster) → **A**ction
> (clean ➜ tokenise ➜ normalise ➜ TF-IDF ➜ model + validation) → **R**esult (numeric success
> criteria and which words/features actually drove it).

### 🚨 The 5 traps interviewers love to check
1. **Leakage** — fit the vectoriser/scaler on TRAIN only; never on the whole dataset.
2. **Bag ignores order & negation** — `"not good"` looks exactly like `"good"`.
3. **Vocabulary on test** — unseen words in test get 0; keep features consistent across splits.
4. **Accuracy under imbalance** — use precision/recall/F1/AUC when classes are skewed.
5. **Scale & sparse memory** — TF-IDF matrices are sparse; use sparse-friendly estimators.

You now know enough to *explain* a full text pipeline end-to-end and defend every choice.
Good luck! 🚀
